In [5]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Mengundih dataset California Housing
X, y = fetch_california_housing(return_X_y=True)

# Membagi dataset menjadi training set dan testing set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Melakukan scaling pada data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Jumlah data training: {X_train.shape}")
print(f"Jumlah data testing: {X_test.shape}")

Jumlah data training: (14448, 8)
Jumlah data testing: (6192, 8)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Inisialisasi model Random Forest Regressor
rf = RandomForestRegressor(random_state=42, n_jobs=1)
rf.fit(X_train, y_train)

# Evaluasi awal model tanpa tuning
y_pred = rf.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error sebelum tuning: {mse:.2f}")


Mean Squared Error sebelum tuning: 0.26


In [ ]:
import time
from sklearn.model_selection import GridSearchCV
from joblib import parallel_backend

start_time = time.time()

# Definisikan parameter grid untuk tuning
# (diperkecil dari 162 kandidat agar lebih realistis & hemat disk)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

# Inisialisasi GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

with parallel_backend("threading"):
    grid_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters: {grid_search.best_params_}")
best_rf_grid = grid_search.best_estimator_

# Evaluasi model terbaik dari GridSearchCV
y_pred_grid = best_rf_grid.predict(X_test)
mse_grid = mean_squared_error(y_test, y_pred_grid)
print(f"Mean Squared Error setelah tuning: {mse_grid:.2f}")

end_time = time.time()
execution_time = end_time - start_time
print(f"Waktu eksekusi GridSearchCV: {execution_time:.2f} detik")


Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   6.3s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   6.9s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   7.3s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  12.9s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   6.2s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  12.6s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  12.3s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   6.0s
[CV

In [12]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

start_time = time.time()

# Definisikan distribusi parameter untuk RandomizedSearchCV
param_dist = {
    'n_estimators': np.arange(100, 500, 100),
    'max_depth': [None] + list(np.arange(10, 50, 10)),
    'min_samples_split': np.arange(2, 11, 2),
    'min_samples_leaf': np.arange(1, 5),
    'bootstrap': [True, False]
}

# Inisialisasi RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=5, cv=3, n_jobs=-1, verbose=2, random_state=42)
random_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (RandomizedSearchCV): {random_search.best_params_}")
best_rf_random = random_search.best_estimator_

# Evaluasi model terbaik dari RandomizedSearchCV
y_pred_random = best_rf_random.predict(X_test)
mse_random = mean_squared_error(y_test, y_pred_random)
print(f"Mean Squared Error setelah tuning (RandomizedSearchCV): {mse_random:.2f}")

end_time = time.time()
execution_time = end_time - start_time
print(f"Waktu eksekusi RandomizedSearchCV: {execution_time:.2f} detik")

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time=  16.0s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time=  16.2s
[CV] END bootstrap=True, max_depth=10, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time=  17.1s
[CV] END bootstrap=True, max_depth=30, min_samples_leaf=2, min_samples_split=6, n_estimators=300; total time=  19.6s
[CV] END bootstrap=False, max_depth=None, min_samples_leaf=2, min_samples_split=8, n_estimators=400; total time=  42.7s
[CV] END bootstrap=True, max_depth=30, min_samples_leaf=2, min_samples_split=6, n_estimators=300; total time=  19.1s
[CV] END bootstrap=False, max_depth=None, min_samples_leaf=2, min_samples_split=8, n_estimators=400; total time=  40.9s
[CV] END bootstrap=False, max_depth=None, min_samples_leaf=2, min_samples_split=8, n_estimators=400; total time=  4

In [ ]:
from skopt import BayesSearchCV
 
start_time = time.time()  # Mencatat waktu mulai
 
# Definisikan ruang pencarian untuk Bayesian Optimization
param_space = {
    'n_estimators': (100, 500),
    'max_depth': (10, 50),
    'min_samples_split': (2, 10),
    'min_samples_leaf': (1, 4),
    'bootstrap': [True, False]
}
 
# Inisialisasi BayesSearchCV
bayes_search = BayesSearchCV(estimator=rf, search_spaces=param_space, n_iter=32, cv=3, n_jobs=1, verbose=2, random_state=42)

with parallel_backend("threading"):
    bayes_search.fit(X_train, y_train)
 
# Output hasil terbaik
print(f"Best parameters (Bayesian Optimization): {bayes_search.best_params_}")
best_rf_bayes = bayes_search.best_estimator_
 
# Evaluasi performa model setelah Random Search
y_pred_bayes = best_rf_bayes.predict(X_test)
bayes_mse = mean_squared_error(y_test, y_pred_bayes)
print(f"MSE after Bayesian Optimization: {bayes_mse:.2f}")
end_time = time.time()  # Mencatat waktu selesai
execution_time = end_time - start_time  # Menghitung selisih waktu
print(f"Waktu eksekusi: {execution_time:.4f} detik")